In [3]:
import torch
import transformers
import itertools
model = transformers.BertModel.from_pretrained('google-bert/bert-base-multilingual-cased', cache_dir="/data/user_data/jiaruil5/.cache/")
tokenizer = transformers.BertTokenizer.from_pretrained('google-bert/bert-base-multilingual-cased', cache_dir="/data/user_data/jiaruil5/.cache/")

In [17]:
src = '- Compared to highly-optimized tiny CNN object detectors, YOLOS achieves competitive performance in terms of AP, FLOPs and FPS. It could serve as a promising starting point for Transformer-based model scaling in object detection. Handling of Variable Input Sizes: \n - Unlike image classification, object detection benchmarks usually have variable image resolutions and aspect ratios.'
tgt = '- مقارنةً بكاشفات الكائنات الصغيرة المحسّنة بشكل كبير، تحقق YOLOS أداءً تنافسياً من حيث AP وFLOPs وFPS. يمكن أن تكون نقطة انطلاق واعدة لتوسيع النماذج المعتمدة على المحولات في كشف الكائنات. التعامل مع أحجام الإدخال المتغيرة:\n- على عكس تصنيف الصور، عادةً ما تحتوي معايير كشف الكائنات على دقة صور وأبعاد متغيرة.'

In [19]:
# pre-processing
# import jieba
# sent_src, sent_tgt = src.strip().split(), [i for i in jieba.cut(tgt, cut_all=False)]

sent_src, sent_tgt = src.strip().split(), tgt.strip().split()
token_src, token_tgt = [tokenizer.tokenize(word) for word in sent_src], [tokenizer.tokenize(word) for word in sent_tgt]
wid_src, wid_tgt = [tokenizer.convert_tokens_to_ids(x) for x in token_src], [tokenizer.convert_tokens_to_ids(x) for x in token_tgt]
ids_src, ids_tgt = tokenizer.prepare_for_model(list(itertools.chain(*wid_src)), return_tensors='pt', model_max_length=tokenizer.model_max_length, truncation=True)['input_ids'], tokenizer.prepare_for_model(list(itertools.chain(*wid_tgt)), return_tensors='pt', truncation=True, model_max_length=tokenizer.model_max_length)['input_ids']
sub2word_map_src = []
for i, word_list in enumerate(token_src):
  sub2word_map_src += [i for x in word_list]
sub2word_map_tgt = []
for i, word_list in enumerate(token_tgt):
  sub2word_map_tgt += [i for x in word_list]

# alignment
align_layer = 8
threshold = 1e-3
model.eval()
with torch.no_grad():
  out_src = model(ids_src.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]
  out_tgt = model(ids_tgt.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]

  dot_prod = torch.matmul(out_src, out_tgt.transpose(-1, -2))

  softmax_srctgt = torch.nn.Softmax(dim=-1)(dot_prod)
  softmax_tgtsrc = torch.nn.Softmax(dim=-2)(dot_prod)

  softmax_inter = (softmax_srctgt > threshold)*(softmax_tgtsrc > threshold)

align_subwords = torch.nonzero(softmax_inter, as_tuple=False)
align_words = set()
for i, j in align_subwords:
  align_words.add( (sub2word_map_src[i], sub2word_map_tgt[j]) )

# printing
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

for i, j in sorted(align_words):
  print(f'{color.BOLD}{color.BLUE}{sent_src[i]}{color.END}==={color.BOLD}{color.RED}{sent_tgt[j]}{color.END}')

-===-
Compared===مقارنةً
to===بكاشفات
highly-optimized===كبير،
detectors,===بكاشفات
detectors,===الكائنات
detectors,===المحسّنة
detectors,===كبير،
YOLOS===YOLOS
achieves===تحقق
performance===أداءً
in===من
terms===حيث
AP,===AP
AP,===وFLOPs
FLOPs===وFLOPs
and===وFPS.
FPS.===وFPS.
could===يمكن
serve===تكون
a===واعدة
promising===لتوسيع
starting===انطلاق
point===واعدة
for===لتوسيع
model===النماذج
scaling===لتوسيع
scaling===المحولات
in===في
object===الكائنات.
detection.===كشف
detection.===الكائنات.
Handling===التعامل
of===مع
Variable===أحجام
Input===أحجام
Sizes:===المتغيرة:
-===-
Unlike===عكس
image===الصور،
classification,===تصنيف
classification,===الصور،
object===الكائنات
detection===كشف
benchmarks===معايير
usually===عادةً
have===تحتوي
have===على
variable===دقة
resolutions===صور
and===وأبعاد
aspect===وأبعاد
ratios.===وأبعاد
ratios.===متغيرة.


In [16]:
len(model(ids_src.unsqueeze(0), output_hidden_states=True)[2])

13